# TP 7 — Jointures, AQE et déséquilibre### Module 4 — Spark SQL et optimisation**Durée :** 2 heures · **Noté sur 20**---## Ce que vous devez savoir faire à la fin1. **Mesurer** l'écart entre `SortMergeJoin` et `BroadcastHashJoin`, en durée **et** en volume   shufflé.2. Reconnaître une jointure que Spark aurait dû diffuser, et la corriger.3. Observer ce que l'AQE corrige, en le désactivant.4. **Provoquer, mesurer et corriger** un déséquilibre — par deux méthodes.5. Choisir entre ces méthodes, et justifier.## Barème| Exercice | Sujet | Points ||---|---|---|| 1 | Préparation | 1 || 2 | Sort merge contre broadcast | 5 || 3 | Statistiques absentes | 4 || 4 | Ce que fait l'AQE | 4 || 5 | Corriger un déséquilibre | 6 |

---# Exercice 1 — Préparation  *(1 point)*

In [ ]:
from pyspark.sql import SparkSession, functions as Ffrom pyspark.sql.types import DoubleTypeimport timeFILIERE = "if"          # "if" ou "an"UTILISATEUR = "etudiant"spark = (SparkSession.builder         .appName("TP7 - Jointures et skew")         .master("local[4]")         .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:8020")         .config("spark.sql.shuffle.partitions", "16")         .getOrCreate())spark.sparkContext.setLogLevel("WARN")print("Spark", spark.version, "| UI :", spark.sparkContext.uiWebUrl)def chrono(libelle, fonction):    debut = time.time()    resultat = fonction()    duree = time.time() - debut    print(f"{libelle:<38} {duree:7.2f} s")    return duree, resultat

In [ ]:
# 1.2 — Une table de faits et un référentielCLE_JOINTURE = "id_marchand" if FILIERE == "if" else "id_oeuvre"CLE_SKEW     = "id_compte"   if FILIERE == "if" else "id_utilisateur"COL_NUM      = "montant"     if FILIERE == "if" else "position_s"REF_CSV      = "if_marchands.csv" if FILIERE == "if" else "an_oeuvres.csv"faits = spark.read.parquet(    f"hdfs://namenode:8020/user/{UTILISATEUR}/formats/pq_trie").cache()referentiel = (spark.read.option("header", True).option("inferSchema", True)               .csv(f"file:///home/tinku/work/data/{REF_CSV}"))print(f"faits       : {faits.count():,} lignes")print(f"referentiel : {referentiel.count():,} lignes")referentiel.printSchema()

In [ ]:
# 1.3 — Écrire le référentiel en Parquet, pour avoir des statistiquesCHEMIN_REF = f"hdfs://namenode:8020/user/{UTILISATEUR}/ref_parquet"referentiel.write.mode("overwrite").parquet(CHEMIN_REF)ref = spark.read.parquet(CHEMIN_REF)import subprocesstaille = subprocess.run(["hdfs", "dfs", "-du", "-s", CHEMIN_REF],                        capture_output=True, text=True).stdout.split()[0]print(f"taille du referentiel en Parquet : {int(taille)/1024**2:.2f} Mio")print("seuil de diffusion :",      int(spark.conf.get('spark.sql.autoBroadcastJoinThreshold'))/1024**2, "Mio")

---# Exercice 2 — Sort merge contre broadcast  *(5 points)*

## 2.1 — **PRÉDICTION** *(1 pt)*Le référentiel fait quelques mégaoctets, bien en dessous du seuil de 10 Mio.Avant d'exécuter : quelle stratégie Spark va-t-il choisir spontanément ? Et si l'on force un`SortMergeJoin`, quel écart attendez-vous — en **durée** et en **volume shufflé** ?

**Votre réponse :***(rédigez ici)*

In [ ]:
# 2.2 — Version forcée en sort mergespark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)jointure_smj = faits.join(ref, CLE_JOINTURE)print("=== plan force en sort merge ===")jointure_smj.explain(mode="formatted")spark.sparkContext.setJobDescription("A - sort merge join")t_smj, _ = chrono("SortMergeJoin", lambda: jointure_smj.count())

In [ ]:
# 2.3 — Version avec diffusionspark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)jointure_bhj = faits.join(F.broadcast(ref), CLE_JOINTURE)print("=== plan avec diffusion ===")jointure_bhj.explain(mode="formatted")spark.sparkContext.setJobDescription("B - broadcast hash join")t_bhj, _ = chrono("BroadcastHashJoin", lambda: jointure_bhj.count())print(f"\nrapport de duree : {t_smj/t_bhj:.1f}x")

### Q2 *(4 pts)* — Ouvrez la Spark UI, onglet **SQL**, et comparez les deux requêtes.- **a.** Quel opérateur de jointure dans chaque plan ? Combien d'`Exchange` dans chacun ?- **b.** Relevez le **Shuffle Write total** de chaque job. Quel rapport ?- **c.** Comparez ce rapport à celui des durées. Lequel est le plus élevé, et pourquoi ?- **d.** Sur un cluster de 40 machines reliées en 10 Gb/s, laquelle des deux métriques  prédirait le mieux le gain réel ? Justifiez.

**Votre réponse :***(rédigez ici)*

---# Exercice 3 — Statistiques absentes  *(4 points)***Objectif.** Reproduire le cas 1 du TD4 : une petite table que Spark ne diffuse pas.

In [ ]:
# 3.1 — Le référentiel devient le RÉSULTAT d'un calcul# Sa taille réelle est minuscule, mais Spark ne peut pas la connaître à l'avance.ref_calcule = (faits.groupBy(CLE_JOINTURE)                    .agg(F.count("*").alias("nb_operations"),                         F.avg(COL_NUM).alias("montant_moyen")))print("lignes du referentiel calcule :", ref_calcule.count())jointure_naive = faits.join(ref_calcule, CLE_JOINTURE)print("\n=== plan SANS broadcast explicite ===")jointure_naive.explain(mode="formatted")

In [ ]:
# 3.2 — Le même, avec broadcast explicitejointure_forcee = faits.join(F.broadcast(ref_calcule), CLE_JOINTURE)print("=== plan AVEC broadcast explicite ===")jointure_forcee.explain(mode="formatted")

In [ ]:
# 3.3 — Mesurerspark.sparkContext.setJobDescription("C - stats absentes, sans broadcast")t_naif, _ = chrono("sans broadcast explicite", lambda: jointure_naive.count())spark.sparkContext.setJobDescription("D - stats absentes, avec broadcast")t_force, _ = chrono("avec broadcast explicite", lambda: jointure_forcee.count())print(f"\nrapport : {t_naif/t_force:.1f}x")

### Q3 *(4 pts)* —- **a.** Quelle stratégie Spark a-t-il choisie **sans** `broadcast()` explicite ? Pourquoi,  alors que le référentiel calculé ne fait que quelques milliers de lignes ?- **b.** Quelle stratégie **avec** ? Quel écart de durée et de volume shufflé ?- **c.** L'AQE aurait-il pu corriger cela tout seul ? Regardez si le plan final mentionne  `AdaptiveSparkPlan isFinalPlan=true` et une conversion. Expliquez le mécanisme.- **d.** Dans quel cas `broadcast()` explicite serait-il **dangereux** ?

**Votre réponse :***(rédigez ici)*

---# Exercice 4 — Ce que fait l'AQE  *(4 points)***Objectif.** Le désactiver pour voir ce qu'il corrigeait.

In [ ]:
# 4.1 — Une requête qui produit beaucoup de partitions presque videsspark.conf.set("spark.sql.shuffle.partitions", "400")def requete_agregee():    return (faits.groupBy(CLE_JOINTURE)                 .agg(F.sum(COL_NUM).alias("total"))                 .filter(F.col("total") > 0)                 .count())# Sans AQEspark.conf.set("spark.sql.adaptive.enabled", "false")spark.sparkContext.setJobDescription("E - SANS AQE")t_sans, _ = chrono("sans AQE (400 partitions figees)", requete_agregee)

In [ ]:
# 4.2 — Avec AQEspark.conf.set("spark.sql.adaptive.enabled", "true")spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")spark.sparkContext.setJobDescription("F - AVEC AQE")t_avec, _ = chrono("avec AQE (fusion automatique)", requete_agregee)print(f"\nrapport : {t_sans/t_avec:.2f}x")

### Q4 *(4 pts)* — Comparez les deux jobs dans la Spark UI.- **a.** Combien de tâches le stage d'agrégation compte-t-il **sans** AQE ? Et **avec** ?- **b.** Quelle est la durée médiane d'une tâche dans chaque cas ?- **c.** Quel mécanisme de l'AQE est à l'œuvre ici ? Décrivez-le en une phrase.- **d.** Quelle conséquence pratique cela a-t-il sur le soin à apporter au réglage de  `spark.sql.shuffle.partitions` ?

**Votre réponse :***(rédigez ici)*

---# Exercice 5 — Corriger un déséquilibre  *(6 points)***Objectif.** Le cas le plus fréquent en production, corrigé par deux méthodes.

In [ ]:
# 5.1 — Mesurer la distributionspark.conf.set("spark.sql.shuffle.partitions", "16")distribution = (faits.groupBy(CLE_SKEW).count()                     .orderBy(F.desc("count")))top = distribution.limit(10).collect()total = faits.count()print("Top 10 des cles :")for r in top:    print(f"  {r[CLE_SKEW]}  {r['count']:>8,}  ({100*r['count']/total:.2f} %)")part_top = sum(r["count"] for r in top)print(f"\nLes 10 premieres cles : {100*part_top/total:.1f} % du volume")

In [ ]:
# 5.2 — L'agrégation naïve sur la clé déséquilibréespark.sparkContext.setJobDescription("G - agregation naive")t_naif, res_naif = chrono("agregation naive",    lambda: (faits.groupBy(CLE_SKEW)                  .agg(F.sum(COL_NUM).alias("total"))                  .count()))print("cles distinctes :", res_naif)print("\nRelevez dans la Spark UI, stage du shuffle :")print("  Summary Metrics -> Duration : mediane et max")print("  Summary Metrics -> Shuffle Read Size : mediane et max")

### Q5a *(2 pts)* — Relevez médiane et maximum des durées de tâches et du *Shuffle Read*.Quel rapport ? Le seuil de 5 est-il dépassé ?

**Votre réponse :***(rédigez ici)*

In [ ]:
# 5.3 — MÉTHODE 1 : isoler les clés chaudesSEUIL = int(total * 0.005)      # cles representant plus de 0,5 % du volumechaudes = [r[CLE_SKEW] for r in           distribution.filter(F.col("count") > SEUIL).collect()]print(f"{len(chaudes)} cles chaudes (> {SEUIL:,} lignes)")def isoler():    froid = faits.filter(~F.col(CLE_SKEW).isin(chaudes))    chaud = faits.filter(F.col(CLE_SKEW).isin(chaudes)).repartition(64, CLE_SKEW)    a = froid.groupBy(CLE_SKEW).agg(F.sum(COL_NUM).alias("total"))    b = chaud.groupBy(CLE_SKEW).agg(F.sum(COL_NUM).alias("total"))    return a.union(b).count()spark.sparkContext.setJobDescription("H - isoler les cles chaudes")t_isole, n_isole = chrono("methode 1 : isoler", isoler)

In [ ]:
# 5.4 — MÉTHODE 2 : saler la cléN_SEL = 32def saler():    sale = faits.withColumn("cle_salee",             F.concat_ws("#", F.col(CLE_SKEW), (F.rand() * N_SEL).cast("int")))    partiel = sale.groupBy("cle_salee").agg(F.sum(COL_NUM).alias("partiel"))    return (partiel            .withColumn(CLE_SKEW, F.split("cle_salee", "#")[0])            .groupBy(CLE_SKEW).agg(F.sum("partiel").alias("total"))            .count())spark.sparkContext.setJobDescription("I - saler la cle")t_sale, n_sale = chrono("methode 2 : saler", saler)print(f"\nnaive  : {t_naif:6.2f} s  ({res_naif} cles)")print(f"isoler : {t_isole:6.2f} s  ({n_isole} cles)  -> {t_naif/t_isole:.2f}x")print(f"saler  : {t_sale:6.2f} s  ({n_sale} cles)  -> {t_naif/t_sale:.2f}x")

### Q5b *(4 pts)* —- **a.** Les trois méthodes donnent-elles le **même nombre de clés** ? Si non, cherchez  l'erreur avant de continuer.- **b.** Quel gain chaque méthode apporte-t-elle ? Comparez aussi le nombre d'`Exchange` de  chaque plan.- **c.** Le salage ajoute un shuffle supplémentaire. Pourquoi peut-il malgré tout être plus  rapide ?- **d.** **Laquelle des deux choisiriez-vous** pour un traitement de production exécuté chaque  nuit ? Justifiez par au moins deux critères, dont un qui n'est pas la performance.

**Votre réponse :***(rédigez ici)*

---# Synthèse| Mesure | Durée | Volume shufflé | Commentaire ||---|---|---|---|| Sort merge join | | | || Broadcast hash join | | | || Sans `broadcast()` sur table calculée | | | || Avec `broadcast()` explicite | | | || Agrégation naïve (déséquilibrée) | | | || Isoler les clés chaudes | | | || Saler la clé | | | |**Question de conclusion.** En reprenant la hiérarchie des gains du cours — lire moins,shuffler moins, équilibrer, régler — à quel niveau se situe chacune des corrections que vousavez mesurées aujourd'hui ?

In [ ]:
faits.unpersist()spark.stop()print("Session fermée.")

---## Avant de rendre- [ ] La **prédiction** de 2.1 est écrite avant exécution.- [ ] Les questions **Q2 à Q5** sont rédigées, avec vos chiffres relevés dans la Spark UI.- [ ] Vous avez vérifié que les trois méthodes de l'exercice 5 donnent le **même résultat**.- [ ] Le tableau de synthèse est complété.- [ ] Notebook exporté en HTML et déposé.**Bon TP.**